In [ ]:
-- ============================================================
-- 03_observability_queries.ipynb
-- Consultas de validación sobre event logs del pipeline
-- Fuente: fintech_finpay.observability
-- ============================================================

-- Registros procesados por capa
SELECT
    origin.flow_name                AS tabla,
    SUM(details.num_output_rows)    AS registros_procesados,
    SUM(details.num_output_rows - COALESCE(details.num_dropped_records, 0)) AS registros_ok,
    SUM(COALESCE(details.num_dropped_records, 0)) AS registros_fallidos,
    MAX(timestamp)                  AS ultima_ejecucion
FROM fintech_finpay.observability.event_log
WHERE event_type = 'flow_progress'
GROUP BY origin.flow_name
ORDER BY tabla;

-- Registros fallidos por expectativa de calidad
SELECT
    origin.flow_name                AS tabla,
    details.expectations.name       AS regla,
    SUM(details.expectations.num_failing_records) AS registros_fallidos,
    MAX(timestamp)                  AS ultima_ejecucion
FROM fintech_finpay.observability.event_log
WHERE event_type = 'flow_progress'
  AND details.expectations IS NOT NULL
GROUP BY origin.flow_name, details.expectations.name
ORDER BY registros_fallidos DESC;

-- Historial de ejecuciones del pipeline
SELECT
    id                  AS pipeline_id,
    origin.update_id    AS update_id,
    event_type,
    message,
    timestamp
FROM fintech_finpay.observability.event_log
ORDER BY timestamp DESC
LIMIT 100;